In [3]:
# Importing the libraries

import os
from typing import TypedDict

from dotenv import load_dotenv
from openai import OpenAI

from langgraph.graph import StateGraph,START,END
load_dotenv()

client=OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [4]:
#Defining the Workflow state
class AgentState(TypedDict):
    question: str
    documents: list[str]
    draft: str
    feedback: str
    revision_count: int
    max_revisions: int
    approved: bool

In [5]:
#Craeting a small knowledge base
knowledge_base = [
    {
        "topic": "RAG",
        "text": (
            "Retrieval-Augmented Generation (RAG) combines "
            "document retrieval with language generation. "
            "It retrieves relevant information and supplies "
            "it to an LLM as context."
        )
    },
    {
        "topic": "Embeddings",
        "text": (
            "Embeddings represent text as numerical vectors. "
            "They help systems compare text by semantic similarity."
        )
    },
    {
        "topic": "Vector databases",
        "text": (
            "Vector databases store embeddings and support "
            "similarity searches to find relevant information."
        )
    },
    {
        "topic": "Chunking",
        "text": (
            "Chunking divides large documents into smaller pieces. "
            "This makes relevant information easier to retrieve."
        )
    }
]

In [6]:
#Creating search node
def search_node(state: AgentState):

    question_words = set(
        state["question"].lower().split()
    )

    results = []

    for document in knowledge_base:
        text = document["text"].lower()

        score = sum(
            word in text
            for word in question_words
        )

        if score > 0:
            results.append(document["text"])

    return {
        "documents": results
    }

In [12]:
#Draft generation node
def draft_node(state: AgentState):

    context = "\n".join(
        state["documents"]
    )

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=(
            "Answer the question using only the supplied "
            "documents. Be clear and informative. "
            "Do not invent unsupported facts."
        ),
        input=f"""
        Question:
        {state["question"]}

        Documents:
        {context}
        """
    )

    return {
        "draft": response.output_text,
        "feedback": "",
        "approved": False
    }

In [8]:
#Review Node
def review_node(state: AgentState):

    draft = state["draft"]
    documents = state["documents"]

    if not documents:
        return {
            "approved": False,
            "feedback": "No supporting documents were retrieved."
        }

    if len(draft.split()) < 35:
        return {
            "approved": False,
            "feedback": (
                "The answer is too short. "
                "Explain the topic in more detail."
            )
        }

    return {
        "approved": True,
        "feedback": "The draft passed the basic quality check."
    }

In [9]:
#Creating the Revision Node
def revision_node(state: AgentState):

    context = "\n".join(
        state["documents"]
    )

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=(
            "Improve the answer using only the supplied "
            "documents. Follow the review feedback. "
            "Do not invent unsupported information."
        ),
        input=f"""
        Original question:
        {state["question"]}

        Source documents:
        {context}

        Previous draft:
        {state["draft"]}

        Review feedback:
        {state["feedback"]}

        Write an improved answer.
        """
    )

    return {
        "draft": response.output_text,
        "revision_count": state["revision_count"] + 1
    }

In [10]:
#Creating the routing function
def route_after_review(state: AgentState):

    if state["approved"]:
        return "finish"

    if state["revision_count"] >= state["max_revisions"]:
        return "finish"

    return "revise"

In [13]:
#Langgraph Workflow
workflow = StateGraph(AgentState)

# Register nodes
workflow.add_node("search", search_node)
workflow.add_node("draft", draft_node)
workflow.add_node("review", review_node)
workflow.add_node("revise", revision_node)

# Starting point
workflow.add_edge(START, "search")

# Generate the initial draft
workflow.add_edge("search", "draft")

# Review the draft
workflow.add_edge("draft", "review")

# Decide what happens after review
workflow.add_conditional_edges(
    "review",
    route_after_review,
    {
        "revise": "revise",
        "finish": END
    }
)

# After revision, review again
workflow.add_edge("revise", "review")

# Compile the graph
app = workflow.compile()

In [14]:
#Running the Workflow
initial_state = {
    "question": "Explain RAG and how it works.",
    "documents": [],
    "draft": "",
    "feedback": "",
    "revision_count": 0,
    "max_revisions": 2,
    "approved": False
}

result = app.invoke(initial_state)

In [15]:
print("Question:")
print(result["question"])

print("\nFinal answer:")
print(result["draft"])

print("\nApproved:")
print(result["approved"])

print("\nRevisions:")
print(result["revision_count"])

print("\nReview feedback:")
print(result["feedback"])

Question:
Explain RAG and how it works.

Final answer:
Retrieval-Augmented Generation (RAG) is a technique that integrates document retrieval with language generation. It works by first retrieving relevant information from a collection of documents and then providing this information as context to a large language model (LLM) to generate responses. 

The process involves representing text as embeddings, which are numerical vectors that capture the semantic meaning of the text. These embeddings enable the system to compare texts based on their semantic similarity. The embeddings are stored in vector databases, which facilitate similarity searches to efficiently find relevant information. By combining retrieval of pertinent documents with generation capabilities, RAG enhances the ability of LLMs to produce informed and contextually accurate outputs.

Approved:
True

Revisions:
0

Review feedback:
The draft passed the basic quality check.


In [16]:
for step in app.stream(initial_state):
    for node_name, node_state in step.items():
        print("\nNode:", node_name)
        print("Updated fields:", list(node_state.keys()))


Node: search
Updated fields: ['documents']

Node: draft
Updated fields: ['draft', 'feedback', 'approved']

Node: review
Updated fields: ['approved', 'feedback']
